In [6]:
import gc
import os
import shutil
import time
import pandas as pd
import xarray as xr

# Storage directories
DIRS = {
    "model_3d": os.path.join("data", "1_model_grid_3d"),
    "in_situ": os.path.join("data", "2_in_situ_observations"),
    "sst": os.path.join("data", "3_sst"),
    "chlorophyll": os.path.join("data", "4_chlorophyll"),
}

for path in DIRS.values():
    os.makedirs(path, exist_ok=True)


def download_gridded_dataset(dataset_id: str, output_filepath: str, chunk_size: int = 5):
    """
    Downloads multi-dimensional ERDDAP gridded datasets in chunks to bypass
    payload size limits, merges them, and cleans up temp files safely on Windows.
    """
    if os.path.exists(output_filepath):
        print(f"[✓] Output already exists, skipping: {output_filepath}")
        return

    opendap_url = f"https://erddap.incois.gov.in/erddap/griddap/{dataset_id}"
    print(f"\n[+] Connecting to {dataset_id} via OPeNDAP...")

    try:
        ds = xr.open_dataset(opendap_url)
    except Exception as e:
        print(f"[!] Failed to connect to {opendap_url}: {e}")
        return

    total_times = len(ds.time)
    print(f"    Total time steps detected: {total_times}")

    temp_dir = os.path.join(os.path.dirname(output_filepath), f"temp_{dataset_id}")
    os.makedirs(temp_dir, exist_ok=True)

    chunk_files = []

    for i in range(0, total_times, chunk_size):
        end_idx = min(i + chunk_size, total_times)
        chunk_file = os.path.join(temp_dir, f"chunk_{i:04d}_{end_idx:04d}.nc")

        if not os.path.exists(chunk_file):
            print(f"    Downloading time index range [{i}:{end_idx}] ...")
            sub_ds = ds.isel(time=slice(i, end_idx))
            sub_ds.to_netcdf(chunk_file)
            sub_ds.close()

        chunk_files.append(chunk_file)

    # Close remote dataset connection
    ds.close()
    del ds
    gc.collect()

    print(f"[*] Merging {len(chunk_files)} chunks into {output_filepath} ...")
    full_ds = xr.open_mfdataset(chunk_files, combine="by_coords")
    full_ds.to_netcdf(output_filepath)

    # Explicitly close and release file locks for Windows OS
    full_ds.close()
    del full_ds
    gc.collect()
    time.sleep(1)

    # Safe directory cleanup
    try:
        shutil.rmtree(temp_dir, ignore_errors=True)
    except Exception as e:
        print(f"    [!] Note: Temp directory cleanup deferred: {e}")

    print(f"[✓] Successfully completed: {output_filepath}")


def download_argo_tabular(start_year: int = 2018, end_year: int = 2026):
    """
    Downloads historical Indian Ocean Argo float profiles year-by-year 
    and concatenates them into one combined CSV table.
    """
    output_csv = os.path.join(DIRS["in_situ"], "Indian_ARGO_Floats_FULL.csv")

    if os.path.exists(output_csv):
        print(f"[✓] Output already exists, skipping: {output_csv}")
        return

    print(f"\n[+] Downloading TableDAP: Indian_ARGO_Floats ({start_year}-{end_year})...")
    yearly_dfs = []

    for year in range(start_year, end_year + 1):
        print(f"    Fetching float profiles for year {year}...")
        url = (
            f"https://erddap.incois.gov.in/erddap/tabledap/Indian_ARGO_Floats.csv?"
            f"time,latitude,longitude,pres,temp,psal"
            f"&time>={year}-01-01T00:00:00Z&time<{year+1}-01-01T00:00:00Z"
        )
        try:
            # Skip second row (index 1) to remove ERDDAP units text
            df = pd.read_csv(url, skiprows=[1], low_memory=False)
            if not df.empty:
                yearly_dfs.append(df)
                print(f"      Retrieved {len(df)} records for {year}.")
        except Exception as e:
            print(f"      [!] Warning: No data or error for year {year}: {e}")

    if yearly_dfs:
        full_df = pd.concat(yearly_dfs, ignore_index=True)
        full_df.to_csv(output_csv, index=False)
        print(f"[✓] Saved complete Argo table ({len(full_df)} total records) to: {output_csv}")
    else:
        print("[!] No tabular Argo data could be compiled.")


if __name__ == "__main__":
    print("====================================================")
    print("  INCOIS ERDDAP Complete Dataset Ingestion Engine   ")
    print("====================================================")

    # 1. 3D Volumetric Ocean Model (Temperature & Salinity vs Depth)
    download_gridded_dataset(
        dataset_id="incois_argo_10day_McCreary",
        output_filepath=os.path.join(DIRS["model_3d"], "incois_argo_10day_McCreary_FULL.nc"),
        chunk_size=5,
    )

    # 2. In-Situ Observations (Argo Float Profiles)
    download_argo_tabular(start_year=2018, end_year=2026)

    # 3. Weekly Sea Surface Temperature (SST) Grid
    download_gridded_dataset(
        dataset_id="incois_argo_sst_weekly",
        output_filepath=os.path.join(DIRS["sst"], "incois_argo_sst_weekly_FULL.nc"),
        chunk_size=10,
    )

    # 4. Chlorophyll Concentration Grid (Ocean Color / BGC)
    download_gridded_dataset(
        dataset_id="IRS_chlorophyll_datasets",
        output_filepath=os.path.join(DIRS["chlorophyll"], "IRS_chlorophyll_datasets_FULL.nc"),
        chunk_size=10,
    )

    print("\n====================================================")
    print("  All Datasets Downloaded and Verified Successfully  ")
    print("====================================================")

  INCOIS ERDDAP Complete Dataset Ingestion Engine   
[✓] Output already exists, skipping: data\1_model_grid_3d\incois_argo_10day_McCreary_FULL.nc

[+] Downloading TableDAP: Indian_ARGO_Floats (2018-2026)...
    Fetching float profiles for year 2018...
      [!] Warning: No data or error for year 2018: HTTP Error 400: 
    Fetching float profiles for year 2019...
      [!] Warning: No data or error for year 2019: HTTP Error 400: 
    Fetching float profiles for year 2020...
      [!] Warning: No data or error for year 2020: HTTP Error 400: 
    Fetching float profiles for year 2021...
      [!] Warning: No data or error for year 2021: HTTP Error 400: 
    Fetching float profiles for year 2022...
      [!] Warning: No data or error for year 2022: HTTP Error 400: 
    Fetching float profiles for year 2023...
      [!] Warning: No data or error for year 2023: HTTP Error 400: 
    Fetching float profiles for year 2024...
      [!] Warning: No data or error for year 2024: HTTP Error 400: 
  

In [7]:
import pandas as pd
import requests

# Fetch the header structure from ERDDAP's info endpoint
info_url = "https://erddap.incois.gov.in/erddap/info/Indian_ARGO_Floats/index.csv"
try:
    df_info = pd.read_csv(info_url)
    variables = df_info[df_info["Row Type"] == "variable"]["Variable Name"].tolist()
    print("Exact Available Columns:", variables)
except Exception as e:
    print(f"Error fetching metadata: {e}")

Exact Available Columns: ['DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'PLATFORM_TYPE', 'time', 'JULD_QC', 'JULD_LOCATION', 'latitude', 'longitude', 'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC']


In [8]:
import os
import urllib.parse
import pandas as pd

output_dir = os.path.join("data", "2_in_situ_observations")
os.makedirs(output_dir, exist_ok=True)
output_csv = os.path.join(output_dir, "Indian_ARGO_Floats_FULL.csv")

# Exact columns confirmed from INCOIS metadata
SELECTED_COLS = [
    "PLATFORM_NUMBER",
    "CYCLE_NUMBER",
    "time",
    "latitude",
    "longitude",
    "PRES",
    "TEMP",
    "PSAL"
]

var_query = ",".join(SELECTED_COLS)
yearly_dfs = []
start_year, end_year = 2018, 2026

print(f"[+] Downloading Indian_ARGO_Floats ({start_year}-{end_year})...")
print(f"    Variables: {var_query}")

for year in range(start_year, end_year + 1):
    print(f"    Fetching float profiles for year {year}...")
    
    time_start = f"{year}-01-01T00:00:00Z"
    time_end = f"{year + 1}-01-01T00:00:00Z"
    
    # Query parameters with safe URL encoding
    query_string = f"{var_query}&time>={time_start}&time<{time_end}"
    encoded_query = urllib.parse.quote(query_string, safe="&,=")
    url = f"https://erddap.incois.gov.in/erddap/tabledap/Indian_ARGO_Floats.csv?{encoded_query}"
    
    try:
        # Skip row 1 to drop the ERDDAP measurement units line
        df_year = pd.read_csv(url, skiprows=[1], low_memory=False)
        if not df_year.empty:
            yearly_dfs.append(df_year)
            print(f"      Retrieved {len(df_year):,} rows for {year}.")
        else:
            print(f"      No records found for {year}.")
    except Exception as e:
        print(f"      [!] Error for year {year}: {e}")

if yearly_dfs:
    full_df = pd.concat(yearly_dfs, ignore_index=True)
    full_df.to_csv(output_csv, index=False)
    print(f"\n[✓] Successfully compiled {len(full_df):,} float records into: {output_csv}")
else:
    print("\n[!] No yearly batches retrieved. Fetching full unpartitioned stream...")
    url_all = f"https://erddap.incois.gov.in/erddap/tabledap/Indian_ARGO_Floats.csv?{var_query}"
    full_df = pd.read_csv(url_all, skiprows=[1], low_memory=False)
    full_df.to_csv(output_csv, index=False)
    print(f"[✓] Saved {len(full_df):,} float records to: {output_csv}")

[+] Downloading Indian_ARGO_Floats (2018-2026)...
    Variables: PLATFORM_NUMBER,CYCLE_NUMBER,time,latitude,longitude,PRES,TEMP,PSAL
    Fetching float profiles for year 2018...
      Retrieved 1,423,935 rows for 2018.
    Fetching float profiles for year 2019...
      Retrieved 1,336,667 rows for 2019.
    Fetching float profiles for year 2020...
      Retrieved 1,348,984 rows for 2020.
    Fetching float profiles for year 2021...
      Retrieved 844,581 rows for 2021.
    Fetching float profiles for year 2022...
      Retrieved 603,931 rows for 2022.
    Fetching float profiles for year 2023...
      Retrieved 416,723 rows for 2023.
    Fetching float profiles for year 2024...
      Retrieved 480,957 rows for 2024.
    Fetching float profiles for year 2025...
      Retrieved 28,676 rows for 2025.
    Fetching float profiles for year 2026...
      [!] Error for year 2026: HTTP Error 404: 

[✓] Successfully compiled 6,484,454 float records into: data\2_in_situ_observations\Indian_ARGO_